## **Bài 3:** Ứng dụng ANN xây dựng model nhận dang chữ số viết tay

---


## 1. Chuẩn bị và Tiền xử lý dữ liệu

Dữ liệu hình ảnh MNIST thường có 784 cột (tương ứng ảnh 28x28 pixels) và 1 cột nhãn (label) cho biết số đó là số mấy (0-9).

### Các bước cần làm:

1.  **Đọc dữ liệu**: Sử dụng Pandas.
2.  **Tách nhãn và đặc trưng**: Tách cột `label` ra khỏi các cột pixel.
3.  **Chuẩn hóa**: Chia giá trị pixel cho 255 để đưa về khoảng [0, 1].
4.  **One-Hot Encoding**: Chuyển nhãn từ dạng số (ví dụ: `5`) sang vector xác suất (ví dụ: `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`). Điều này bắt buộc cho lớp Output có 10 nơ-ron.

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# 1. Đọc dữ liệu (Lưu ý đường dẫn file)
train_df = pd.read_csv('trainMNIST.csv')
test_df = pd.read_csv('testMNIST.csv')

# Kiểm tra dữ liệu
print(train_df.head())

# 2. Tách Label và Pixel
# Giả sử cột 'label' là cột đầu tiên hoặc có tên là 'label'.
# Bạn cần check tên cột bằng lệnh train_df.columns
# Ở đây giả sử cột nhãn tên là 'label'
y_train_raw = train_df['label'].values
X_train = train_df.drop('label', axis=1).values

# Tương tự cho tập test (lưu ý file testMNIST đôi khi không có nhãn, nếu có thì tách như trên)
if 'label' in test_df.columns:
    y_test_raw = test_df['label'].values
    X_test = test_df.drop('label', axis=1).values
else:
    # Nếu file test không có nhãn, ta chỉ dùng để dự đoán
    X_test = test_df.values

# 3. Chuẩn hóa pixel (0-255 -> 0-1)
X_train = X_train / 255.0
X_test = X_test / 255.0

# 4. One-Hot Encoding cho nhãn (Quan trọng!)
# Ví dụ: số 2 -> [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(y_train_raw.reshape(-1, 1))

if 'label' in test_df.columns:
    y_test = encoder.transform(y_test_raw.reshape(-1, 1))

print("Shape X:", X_train.shape) # (N, 784)
print("Shape y:", y_train.shape) # (N, 10)

   label  pixel0  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  \
0      1       0       0       0       0       0       0       0       0   
1      0       0       0       0       0       0       0       0       0   
2      1       0       0       0       0       0       0       0       0   
3      4       0       0       0       0       0       0       0       0   
4      0       0       0       0       0       0       0       0       0   

   pixel8  ...  pixel774  pixel775  pixel776  pixel777  pixel778  pixel779  \
0       0  ...         0         0         0         0         0         0   
1       0  ...         0         0         0         0         0         0   
2       0  ...         0         0         0         0         0         0   
3       0  ...         0         0         0         0         0         0   
4       0  ...         0         0         0         0         0         0   

   pixel780  pixel781  pixel782  pixel783  
0         0         0         

In [3]:
# Hàm Softmax (Dùng cho Output Layer đa lớp)
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True)) # Trừ max để tránh tràn số (overflow)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# Hàm ReLU (Dùng cho Hidden Layer)
def relu(z):
    return np.maximum(0, z)

# Đạo hàm ReLU
def relu_derivative(z):
    return (z > 0).astype(float)

# Hàm Loss: Categorical Cross-Entropy
def cross_entropy_loss(y_true, y_pred):
    # Cộng thêm epsilon nhỏ xíu để tránh log(0)
    n_samples = y_true.shape[0]
    logp = -np.log(y_pred[range(n_samples), y_true.argmax(axis=1)] + 1e-9)
    loss = np.sum(logp) / n_samples
    return loss

## 3. Huấn luyện (Training Loop)

Cấu trúc mạng gợi ý:

- **Input**: 784 (pixels)
- **Hidden**: 128 (nơ-ron)
- **Output**: 10 (lớp số 0-9)


In [4]:
# Khởi tạo tham số
input_size = 784
hidden_size = 128
output_size = 10
learning_rate = 0.1
epochs = 500

np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

losses = []

for epoch in range(epochs):
    # --- 1. Forward Propagation ---
    Z1 = np.dot(X_train, W1) + b1
    A1 = relu(Z1) # Lớp ẩn dùng ReLU

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2) # Lớp ra dùng Softmax

    # --- 2. Tính Loss ---
    loss = cross_entropy_loss(y_train, A2)
    losses.append(loss)

    # --- 3. Backpropagation ---
    m = X_train.shape[0]

    # Đạo hàm tại Output (Softmax + CrossEntropy rất đẹp: A2 - y)
    dZ2 = A2 - y_train
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

    # Đạo hàm tại Hidden
    dZ1 = np.dot(dZ2, W2.T) * relu_derivative(Z1)
    dW1 = (1/m) * np.dot(X_train.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

    # --- 4. Cập nhật trọng số ---
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 2.3031
Epoch 50, Loss: 1.9658
Epoch 100, Loss: 0.9434
Epoch 150, Loss: 0.6202
Epoch 200, Loss: 0.4990
Epoch 250, Loss: 0.4367
Epoch 300, Loss: 0.3990
Epoch 350, Loss: 0.3736
Epoch 400, Loss: 0.3550
Epoch 450, Loss: 0.3405



## 4. Dự đoán và Đánh giá


In [6]:
# Dự đoán trên tập Test (nếu có nhãn)
if 'label' in test_df.columns:
    # Forward pass
    Z1_test = np.dot(X_test, W1) + b1
    A1_test = relu(Z1_test)
    Z2_test = np.dot(A1_test, W2) + b2
    A2_test = softmax(Z2_test)

    # Chuyển từ xác suất về nhãn (0-9)
    predictions = np.argmax(A2_test, axis=1)
    true_labels = np.argmax(y_test, axis=1) # Chuyển y_test từ one-hot về nhãn gốc

    accuracy = np.mean(predictions == true_labels)
    print(f"Độ chính xác trên tập Test: {accuracy * 100:.2f}%")
else:
    print("Tập test không có nhãn để đánh giá.")

Tập test không có nhãn để đánh giá.
